# Linear SVM for Movie Review Emotion Classification
**Fernando B. K. H. — IT25102631 — IT2011 Progress Review II**

Run on a CPU from the repository root or its notebooks/ directory. In Colab, first open or clone the team repository and change into it. Use `processed_movie_reviews.csv`.

**Model suitability:** LinearSVC learns linear decision functions over sparse text features and supports seven-class classification using one-vs-rest. It is distinct from Logistic Regression, Multinomial Naive Bayes, Random Forest, HistGradientBoosting and MLP. It uses decision scores, not calibrated probabilities. Genres link this contribution to my earlier genre encoding work.

**Dataset check:** supplied CSV has 16,107 reviews, 1,309 movies and seven classes with exactly the class counts printed in the teammates' notebooks. Their source CSV/loader is not attached, so exact row identity is not established. Their pipelines use raw Reviews and genres; this notebook uses the supplied filtered text and already-encoded genres. Feature differences must be reported.

**Comparison status:** teammates use 12,886 training / 3,221 test rows, five movie-grouped CV folds and macro F1 selection. Equal counts or seeds do not establish identical partitions. This notebook accepts a shared row manifest; without one, it makes an explicitly provisional movie-grouped split. Do not rank its results against the teammates' existing scores until the row and fold memberships are verified. Alternatively, all members can rerun on the manifest exported here.

**Lab alignment:** SVM, scaling and evaluation concepts follow the supplied labs. LinearSVC, TF-IDF, grouped splitting and Pipeline are project adaptations, not claimed as exact lab-sheet code. Text features are used here because the teammates also use TF-IDF.

## 1. Imports

In [ ]:
from pathlib import Path
import json, hashlib, warnings, time, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, ConfusionMatrixDisplay)
from sklearn.exceptions import ConvergenceWarning
SEED=42
warnings.filterwarnings('error',category=ConvergenceWarning)
print('scikit-learn:',sklearn.__version__)

## 2. Load the repository dataset

In [ ]:
# Resolve repository root when launched from notebooks/ or repository root.
import os, sys
REPO_ROOT=Path.cwd().resolve()
if REPO_ROOT.name=='notebooks':
    REPO_ROOT=REPO_ROOT.parent
if not (REPO_ROOT/'src').is_dir():
    raise FileNotFoundError('Run this notebook inside the team repository containing src/.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))
DATA_PATH = Path('results/outputs/processed_movie_reviews.csv')
OUT = Path('results/phase2')
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH.resolve()}')
OUT.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATA_PATH)
assert len(df)==16107, 'Unexpected dataset: check the file before training.'
genres=sorted(c for c in df if c.startswith('genre_'))
assert len(genres)==20
required=['movie_name','emotion','cleaned_review','tokens_filtered','word_count']+genres
assert set(required).issubset(df.columns)
assert not df[required].isna().any().any(), 'Inspect missing values before continuing.'
assert df[genres].isin([0,1]).all().all()
assert np.isfinite(df[['word_count']].to_numpy()).all() and df.word_count.ge(0).all()
EXPECTED={'sadness':6555,'joy':2765,'anticipation':2166,'optimism':1772,
          'fear':1386,'anger':929,'disgust':534}
assert df.emotion.value_counts().to_dict()==EXPECTED
assert df.movie_name.nunique()==1309
LABELS=sorted(EXPECTED)
# Content-based IDs survive row reordering and expose changed labels/text.
def row_key(row):
    payload=[str(row[c]) for c in ['movie_name','emotion','cleaned_review']]
    return hashlib.sha256(json.dumps(payload,ensure_ascii=False).encode()).hexdigest()
df['review_key']=df.apply(row_key,axis=1)
assert df.review_key.is_unique
dataset_identity=hashlib.sha256('\n'.join(sorted(df.review_key)).encode()).hexdigest()
print('CSV:',DATA_PATH,'Shape:',df.shape,'Movies:',df.movie_name.nunique())
print(df.emotion.value_counts())
print('Row-set fingerprint:',dataset_identity)
print('Aggregate counts match teammates; exact row identity remains unverified.')

## 3. Shared split or provisional split
Optional: place `team_split_manifest.csv` beside the dataset. Required columns: `review_key`, `split` (train/test), `validation_fold` (0–4 for training; -1 for test). A teammate must export actual memberships with the same row-key definition above. Every row must match exactly. Loading a manifest verifies compatibility with this CSV, not its provenance; confirm it is the team's actual manifest.

Otherwise, a fixed outer five-fold movie-grouped split reserves its first fold as test; inner five-fold movie-grouped validation uses the remainder. Do not search seeds to obtain better scores or matching row counts. The fallback split can have different sizes from the team split.

In [ ]:
MANIFEST=DATA_PATH.parent/'team_split_manifest.csv'
if MANIFEST.exists():
    m=pd.read_csv(MANIFEST)
    assert m.review_key.is_unique and set(m.review_key)==set(df.review_key)
    assert set(m['split'])=={'train','test'}
    df=df.merge(m[['review_key','split','validation_fold']],on='review_key',validate='one_to_one')
    assert set(df.loc[df['split']=='train','validation_fold'])==set(range(5))
    assert (df.loc[df['split']=='test','validation_fold']==-1).all()
    protocol='supplied_manifest_provenance_requires_team_confirmation'
else:
    outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
    tr,te=next(outer.split(df,df.emotion,df.movie_name))
    df['split']='train';df.loc[te,'split']='test';df['validation_fold']=-1
    development=df.loc[tr]
    inner=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
    for fold,(_,va) in enumerate(inner.split(development,development.emotion,development.movie_name)):
        df.loc[development.index[va],'validation_fold']=fold
    protocol='PROVISIONAL_NOT_VERIFIED_AGAINST_TEAM'
train=df[df['split']=='train'].reset_index(drop=True)
test=df[df['split']=='test'].reset_index(drop=True)
folds=[(np.flatnonzero(train.validation_fold!=k),np.flatnonzero(train.validation_fold==k)) for k in range(5)]
def check_separation(a,b):
    assert set(a.movie_name).isdisjoint(b.movie_name), 'Movie overlap detected.'
    for c in ['cleaned_review','tokens_filtered']:
        ta=a[c].str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
        tb=b[c].str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
        assert set(ta).isdisjoint(tb), 'Exact text overlap detected: fix the grouping.'
    assert set(a.emotion)==set(b.emotion)==set(LABELS)
check_separation(train,test)
for tr,va in folds:check_separation(train.iloc[tr],train.iloc[va])
manifest=df[['review_key','movie_name','emotion','split','validation_fold']].sort_values('review_key')
manifest.to_csv(OUT/'linear_svc_split_manifest.csv',index=False)
split_identity=hashlib.sha256(manifest.to_csv(index=False).encode()).hexdigest()
print(protocol)
print('Training/test rows:',len(train),len(test))
print('Training/test movies:',train.movie_name.nunique(),test.movie_name.nunique())
print('Validation sizes:',[len(va) for _,va in folds])
print('Split fingerprint:',split_identity)

## 4. Define preprocessing varieties and Linear SVM
A: filtered-text TF-IDF only. B: text plus binary genres. C: text plus genres and RobustScaler-transformed supplied word count. Ratings are excluded until label provenance and prediction-time availability are established.

TF-IDF: 2,500 maximum vocabulary features, unigrams+bigrams, minimum document frequency 3, sublinear TF (1 + log count), default L2 normalization. These vocabulary settings match the teammates' visible settings, but upstream text cleaning may differ. Pipeline fits vocabulary, IDF and scaling within every fold. Existing filtered text and genres are assumed to be deterministic preprocessing; their upstream provenance still needs checking.

**12 configurations = 3 preprocessing varieties × 2 C values (0.1, 1) × 2 weights (None, balanced). Five folds = 60 CV fits, plus three per-variety refits = 63 fits.** Larger C means weaker regularization. Balanced weights give rare classes more influence. max_iter controls solver convergence, not model capacity.

In [ ]:
def prep(variety):
    branches=[('text',TfidfVectorizer(max_features=2500,ngram_range=(1,2),
                 min_df=3,sublinear_tf=True),'tokens_filtered')]
    if variety in ['B_text_genres','C_text_genres_length']:
        branches.append(('genres','passthrough',genres))
    if variety=='C_text_genres_length':branches.append(('length',RobustScaler(),['word_count']))
    return ColumnTransformer(branches,remainder='drop',sparse_threshold=1.0)
VARIETIES=['A_text','B_text_genres','C_text_genres_length']
SCORING={'macro_f1':'f1_macro','accuracy':'accuracy','weighted_f1':'f1_weighted',
         'macro_precision':'precision_macro','macro_recall':'recall_macro'}
cache=joblib.Memory(tempfile.mkdtemp(prefix='fernando_svm_'),verbose=0)
pipeline=Pipeline([('prep',prep('A_text')),
    ('clf',LinearSVC(dual=False,max_iter=20000,random_state=SEED))],memory=cache)

## 5. Tuning on training data only
Macro F1 is the primary selection metric because each emotion matters. Accuracy is overall correctness; weighted F1 reflects class frequency. Macro precision and recall explain prediction reliability and class coverage. A class with no predicted examples can generate an undefined-metric warning; retain that warning rather than hide all warnings.

All candidates use the same five folds. A/C=1/unweighted is the baseline included in the grid. Each row in the exported table is one distinct configuration. Training macro F1 versus validation macro F1 supports overfitting analysis. Test data is not used to select a configuration.

In [ ]:
tables=[];searches={};start=time.time()
for variety in VARIETIES:
    pipeline.set_params(prep=prep(variety))
    search=GridSearchCV(pipeline,{'clf__C':[0.1,1.0],'clf__class_weight':[None,'balanced']},
        cv=folds,scoring=SCORING,refit="macro_f1",return_train_score=True,n_jobs=1,error_score='raise')
    search.fit(train,train.emotion)
    searches[variety]=search
    table=pd.DataFrame(search.cv_results_);table['variety']=variety;tables.append(table)
    table.to_csv(OUT/('linear_svc_'+variety+'_cv.csv'),index=False)
    print(variety,'complete; elapsed seconds',round(time.time()-start),flush=True)
results=pd.concat(tables,ignore_index=True)
results['C']=results.param_clf__C.astype(float)
results['class_weight']=results.param_clf__class_weight.fillna('none')
results['gap']=results.mean_train_macro_f1-results.mean_test_macro_f1
results=results.sort_values(['mean_test_macro_f1','std_test_macro_f1','C','variety'],
    ascending=[False,True,True,True]).reset_index(drop=True)
columns=['variety','C','class_weight','mean_train_macro_f1','mean_test_macro_f1',
    'std_test_macro_f1','mean_test_accuracy','mean_test_weighted_f1',
    'mean_test_macro_precision','mean_test_macro_recall','gap']
results[columns].to_csv(OUT/'linear_svc_cv_results.csv',index=False)
print(results[columns].to_string(index=False))
best=results.iloc[0]
print('Selected by validation:',best[['variety','C','class_weight']].to_dict())

## 6. Compare configurations and assess overfitting
`mean_test_*` in GridSearchCV means inner validation, not the held-out test set. A high training score with much lower validation performance suggests overfitting. Low scores on both can indicate weak features, noisy labels or insufficient model capacity. A small gap does not prove good performance. Do not use an arbitrary 60% threshold or choose C after viewing test results.

In [ ]:
fig,ax=plt.subplots(figsize=(12,5))
x=np.arange(len(results))
ax.bar(x-.2,results.mean_train_macro_f1,.4,label='Training')
ax.bar(x+.2,results.mean_test_macro_f1,.4,yerr=results.std_test_macro_f1,label='Validation')
labels=[f'{r.variety} C={r.C} {r.class_weight}' for _,r in results.iterrows()]
ax.set_xticks(x,labels,rotation=70,ha='right',fontsize=8);ax.set_ylim(0,1)
ax.set_ylabel('Macro F1');ax.legend();plt.tight_layout()
plt.savefig(OUT/'linear_svc_training_validation.png',dpi=160);plt.show()
print('Selected mean training macro F1:',best.mean_train_macro_f1)
print('Selected mean validation macro F1:',best.mean_test_macro_f1)
print('Gap:',best.gap)

## 7. Select the fitted pipeline
Reuse the best fitted pipeline from the validation-winning preprocessing search. Test evaluation occurs only in the next cell.

In [ ]:
# Each preprocessing search refits its best candidate on the training set.
# Reuse the winning fitted pipeline; do not train or tune on the held-out set.
selected_search=searches[best.variety]
final=selected_search.best_estimator_
print('Selected preprocessing:',best.variety)
print('Selected parameters:',selected_search.best_params_)
print('Selected validation macro F1:',selected_search.best_score_)


## 8. Standardized evaluation and export
Use the group's `src.evaluation` helpers, following the call signatures in the Random Forest notebook. The evaluator creates the prediction table; the export helper writes `results/phase2/linear_svc_results.json` and `results/phase2/linear_svc_test_predictions.csv`. Split and fold SHA-256 fingerprints come from the shared helpers using the actual partitions. The shared validator checks the saved artifact schema.

Matching export format does not establish identical splits. The provisional/shared-manifest status remains recorded. Confirm fingerprints against teammates before official ranking.

This revision requires the repository's actual src/evaluation.py. Its calls were checked against the teammate notebook, but cannot be executed here without that module. Prior run outputs have been cleared to avoid presenting them as results from this revised export flow.

In [ ]:
from src.evaluation import (
    save_model_results, evaluate_on_test, split_fingerprint,
    fold_fingerprints, cv_fold_scores, compute_baselines,
    load_model_results, validate_results,
)

# Use the actual partitions/folds used above; do not invent team fingerprints.
SPLIT_INFO=split_fingerprint(train,test,random_state=SEED,test_size=0.2)
CV_FINGERPRINTS=fold_fingerprints(folds,train['movie_name'])
BASELINES=compute_baselines(train.emotion,test.emotion,n_seeds=200)
cv_payload=cv_fold_scores(selected_search)

# The group's evaluator owns the prediction columns and test-metric schema.
# Evaluate the selected model once on the held-out set.
test_metrics,predictions_df=evaluate_on_test(final,test,test.emotion)
meta={
    'model_label':'Linear SVC',
    'member_id':'IT25102631',
    'algorithm':'LinearSVC',
    'preprocessing':str(best.variety),
    'protocol_status':protocol,
    'tuning':{
        'method':'GridSearchCV',
        'n_configs':int(len(results)),
        'n_folds':len(folds),
        'scoring':list(SCORING.keys()),
        'elapsed_seconds':float(time.time()-start),
        'best_params':selected_search.best_params_,
    },
}
json_path,csv_path=save_model_results(
    out_dir=OUT,
    model_key='linear_svc',
    meta=meta,
    split=SPLIT_INFO,
    cv=cv_payload,
    cv_fingerprints=CV_FINGERPRINTS,
    test=test_metrics,
    baselines=BASELINES,
    predictions_df=predictions_df,
)
validate_results(load_model_results(json_path))
print('Standardized results:',json_path)
print('Standardized predictions:',csv_path)
print('Protocol:',protocol)
print('Test accuracy:',test_metrics['accuracy'])
print('Test macro F1:',test_metrics['macro_f1'])
print('Test weighted F1:',test_metrics['weighted_f1'])

# Plot directly from the group's evaluated confusion matrix.
fig,ax=plt.subplots(figsize=(10,8))
ConfusionMatrixDisplay(confusion_matrix=np.asarray(test_metrics['confusion_matrix']),
                       display_labels=LABELS).plot(ax=ax,cmap='Blues',xticks_rotation=45)
ax.set_title('Linear SVC');plt.tight_layout()
plt.savefig(OUT/'linear_svc_confusion_matrix.png',dpi=160);plt.show()

# Save a reusable fitted model, separate from standardized evaluation artifacts.
joblib.dump({'pipeline':final,'labels':LABELS,'sklearn_version':sklearn.__version__},
            OUT/'linear_svc.joblib')
loaded=joblib.load(OUT/'linear_svc.joblib')['pipeline']
assert np.array_equal(loaded.predict(test.iloc[:5]),final.predict(test.iloc[:5]))
print('Demo predictions:',loaded.predict(test.iloc[:3]))
print('Recorded labels:',test.emotion.iloc[:3].tolist())
cache.clear(warn=False)


## 9. Viva guide and remaining group integration
- **What I built:** a seven-class Linear SVM using the specified processed movie-review dataset.
- **Why this model:** it can learn from sparse numerical text features efficiently; genre features connect to my individual preprocessing work. Linear boundaries and bag-of-words representations limit context understanding.
- **Experiments:** three preprocessing varieties and four C/weight combinations each; 12 configurations, five folds, 60 CV fits and three per-variety refits.
- **Validation:** movie-grouped holdout and movie-grouped five-fold CV. Scaling and TF-IDF are fitted inside each fold. Exact text overlaps are checked as well.
- **Conclusion:** read the selected settings and actual metrics from the executed cells. Explain class-specific errors and the training–validation gap; do not invent a good-fit diagnosis or claim this model wins before comparing compatible runs.
- **Limitations:** labels may be movie-level according to teammates' notes; verify provenance. Genre/text preprocessing from the supplied CSV differs from their raw-text pipeline. Seven classes only, substantial imbalance, possible near-duplicates, a limited tuning grid, and historical exposure to this dataset remain limitations.
- **Possible improvements:** verify review-level annotations, collect minority examples, preserve negation, broaden development-only tuning and use repeated grouped CV. A fresh external dataset is preferable for a new final performance claim.
- **Team comparison:** attach shared split files and rerun, or rerun the other models on this manifest. Use macro F1 first, followed by weighted F1, accuracy, variability and runtime. Record feature differences. Do not rank provisional scores beside the existing Naive Bayes/Random Forest scores as if the partitions were confirmed.

Library reference: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html